# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import time
from pathlib import Path
import re

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path("data")   # adjust if your folder layout differs)

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])


Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [4]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""    
    return [
        {
            "role": "system",
            "content": """
            Extract the role, company, and years of experience required from the job snippet in JSON format.
            """,
        },
        {
            "role": "user",
            "content": f"<job_snippet>{snippet_text}</job_snippet>",
        },
    ]
    


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    return [
        {
            "role": "system",
            "content": """
            Extract the role, company, and years of experience required from the job snippet in JSON format.
            
            Example:
            <job_snippet>Software Engineer at OpenAI, 3 years experience required</job_snippet>
            <output_format>
                {
                    "role": "Software Engineer",
                    "company": "OpenAI",
                    "years_experience_required": 3
                }
            </output_format>
            
            Another Example:
            <job_snippet>Data Scientist at Google</job_snippet>
            <output_format>
                {
                    "role": "Data Scientist",
                    "company": "Google",
                    "years_experience_required": null
                }
            </output_format>
            """,
        },
        {
            "role": "user",
            "content": f"<job_snippet>{snippet_text}</job_snippet>",
        },
    ]



def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {
            "role": "system",
            "content": """
            You are an expert job posting analyst. Extract the role, company, and years of experience required from the job snippet in JSON format.
            
            <output_format>
               {
                   "role": string | null,
                   "company": string | null,
                   "years_experience_required": int | null
               }
            </output_format>
            """,
        },
        {
            "role": "user",
            "content": f"<job_snippet>{snippet_text}</job_snippet>",
        },
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            "role": "system",
            "content": """
            You are an expert job posting analyst. Analyze the job posting carefully and extract the role, company, and years of experience required, perform an internal verification pass to ensure the accuracy of the extracted information. Provide the output in JSON format.
            
            <output_format>
               {
                   "role": string | null,
                   "company": string | null,
                   "years_experience_required": int | null
               }
            </output_format>
            """,
        },
        {
            "role": "user",
            "content": f"<job_snippet>{snippet_text}</job_snippet>",
        },
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [5]:


def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    text = re.sub(r'^```json\s*', '', text.strip())
    text = re.sub(r'\s*```$', '', text)
    try:
        result = json.loads(text)
        return result if isinstance(result, dict) else None
    except json.JSONDecodeError:
        return None


async def run_one(strategy_name: str, snippet) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    # TODO: call the model, time the call, compute cost, parse the response
    prompt_fn = STRATEGIES[strategy_name]          # e.g. prompt_zero_shot
    prompt = prompt_fn(snippet)                    # build the messages list
    start = time.perf_counter()
    response = await client.chat.completions.create(
        model=MODEL, 
        messages=prompt, 
        temperature=0.0
    )
    elapsed = time.perf_counter() - start
    usage = response.usage
    cost = (usage.prompt_tokens  * RATES[MODEL]['in'] +
            usage.completion_tokens * RATES[MODEL]['out'])
    parsed = parse_response(response.choices[0].message.content)
    return {
        'strategy': strategy_name,
        'snippet':  snippet,
        'raw':      response.choices[0].message.content,
        'parsed':   parsed,
        'cost':     cost,
        'elapsed':  elapsed,
    }
    #return response


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # TODO: build the task list, gather, return results
    tasks = [run_one(name, s) for name in STRATEGIES for s in snippets]
    return await asyncio.gather(*tasks)

In [6]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet': {'id': 'j01',
  'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'},
 'raw': '```json\n{\n  "role": "Senior Software Engineer",\n  "company": "Acme Corp",\n  "years_of_experience": "5+"\n}\n```',
 'parsed': {'role': 'Senior Software Engineer',
  'company': 'Acme Corp',
  'years_of_experience': '5+'},
 'cost': 3.45e-05,
 'elapsed': 5.524955000000773}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [18]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""

    #print(f"ext->{extracted['years_experience_required']}") 
    #print(f"gold-->gold.get('years_experience_required','')")
    ext_years = extracted.get('years_experience_required') or extracted.get('years_of_experience', '')
    gold_years = gold.get('years_experience_required', '')
    if extracted is None:
        return 0
    score = 0
    if extracted['role'] and extracted['role'].lower().strip() == gold.get('role', '').lower().strip():
        score += 1
    if (
        extracted['company']
        and extracted['company'].lower().strip() == gold.get('company','').lower().strip()
    ):
        score += 1
    if str(ext_years).strip() == str(gold_years).strip():
        score += 1
    return score


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    extracted_str = json.dumps(extracted)
    system_prompt = """You are a strict, fair evaluator of answers from a question-answering assistant.

    Evaluate these three specific fields:
    1. role
    2. company
    3. years of experience required


Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
   Provide your evaluation in a valid JSON format with two keys:
    - "reasoning": A brief, one-sentence explanation justifying the score based on the rubric.
    - "score": The integer score (1, 2, 3, or 4).
"""
    user_prompt=f"""
    Snippet:{snippet_text}

    Gold answer:
    {json.dumps(gold)}

    Model's extraction:{extracted_str}
    """
    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[ {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}],
        temperature=0.0,
    )
    text = response.choices[0].message.content.strip()
    try:
        # handle JSON response like {"reasoning": "...", "score": 4}
        parsed = json.loads(text)
        return max(1, min(4, int(parsed['score'])))
    except (json.JSONDecodeError, KeyError):
        # fallback: find first digit 1-4 in the text
        for char in text:
            if char in '1234':
                return int(char)
        return 1
   # return response

In [19]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
judge_tasks = [
    score_llm_judge(r['snippet']['snippet'], r['parsed'], golden)
    for r in results
]
judge_scores = await asyncio.gather(*judge_tasks)

scored = []
for r, judge_score in zip(results, judge_scores):
    gold = golden
    scored.append({
        **r,
        'parse_success':   r['parsed'] is not None,
        'accuracy':        score_accuracy(r['parsed'], gold),
        'llm_judge_score': judge_score,
    })
print(scored[0])
print(f'Scored {len(scored)} results.')

{'strategy': 'zero_shot', 'snippet': {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}, 'raw': '```json\n{\n  "role": "Senior Software Engineer",\n  "company": "Acme Corp",\n  "years_of_experience": "5+"\n}\n```', 'parsed': {'role': 'Senior Software Engineer', 'company': 'Acme Corp', 'years_of_experience': '5+'}, 'cost': 3.45e-05, 'elapsed': 5.524955000000773, 'parse_success': True, 'accuracy': 0, 'llm_judge_score': 4}
Scored 40 results.


## Step 5 — Build the comparison table

In [20]:
df = pd.DataFrame(scored)

summary = (
        df.groupby("strategy")
        .agg(
            {
                "accuracy": "mean",
                "parse_success": "mean",
                "llm_judge_score": "mean",
                "cost": "sum",
                "elapsed": "median",
            }
        )
        .round(
            {
                "accuracy": 2,
                "parse_success": 2,
                "llm_judge_score": 2,
                "cost": 6,
                "elapsed": 3,
            }
        )
    )

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
print(summary)

            Accuracy (mean of 3)  Parse rate  Judge score  Total cost ($)  \
strategy                                                                    
cot                          0.2         1.0          3.9        0.000428   
few_shot                     0.2         1.0          3.8        0.000507   
structured                   0.2         1.0          3.9        0.000398   
zero_shot                    0.4         1.0          3.5        0.000357   

            Latency p50 (s)  
strategy                     
cot                   4.103  
few_shot              3.994  
structured            4.131  
zero_shot             4.358  


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```